# Part 1 · Where do people want to go?
### OD weights and how pedestrian flows spread through a network

You'll change **who wants to go where** in a small station and watch *where* and *when* crowds form.
Work in pairs: one person clicks, the other writes down predictions. Swap every exercise.


**How to use this notebook**
* Press **▶** on the left of a grey cell (or click it and press **Shift + Enter**) to run it.
* Cells with sliders and boxes are *forms*: change the values on the right, then press ▶ again.
* You never have to edit code. If you're curious, double-click a form to see the code behind it.
* Something broken? **Runtime → Restart session**, then run the first cell again.

In [ ]:
#@title ▶ **Run me first**: set up the workshop (≈30 s) { display-mode: "form" }
#@markdown Installs PedNStream and the workshop files. Run it again if Colab restarted.
import importlib, os, subprocess, sys

_FILES = {'workshop_tools.py': '"""Helper layer for the PedNStream OD-weights workshop.\n\nStudents call a handful of friendly functions:\n\n    res = run(od={"0_5": 1, "0_6": 1, "1_5": 1, "1_6": 1})\n    res.kpis()            # headline numbers\n    res.plot_map()        # network coloured by density at the worst moment\n    res.plot_links()      # density over time on every corridor\n    res.plot_arrivals()   # who reached which destination, and when\n    compare({"A": res1, "B": res2})\n\nEverything underneath is plain PedNStream (NetworkEnvGenerator + Network).\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Dict, Iterable, Optional\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nfrom matplotlib.collections import LineCollection\nfrom matplotlib.colors import Normalize\n\nfrom pednstream.utils.network_env_generator import NetworkEnvGenerator\n\nHERE = Path(__file__).resolve().parent\nDEFAULT_NETWORK = HERE / "networks" / "station_square"\n\nNODE_NAMES = {\n    0: "West Gate (A)",\n    1: "South Gate (B)",\n    2: "Plaza",\n    3: "Hall",\n    4: "Mall Jct",\n    5: "Platform (X)",\n    6: "City Exit (Y)",\n}\n\nCORRIDORS = ["0_2", "1_3", "2_3", "2_4", "3_4", "3_5", "4_6"]\n\n\n# ----------------------------------------------------------------------------\n# Input helpers\n# ----------------------------------------------------------------------------\ndef _parse_key(key) -> tuple:\n    if isinstance(key, tuple):\n        return tuple(int(k) for k in key)\n    o, d = str(key).split("_")\n    return int(o), int(d)\n\n\ndef switch(before: float, after: float, at_step: int, steps: int = 300) -> np.ndarray:\n    """A time-varying OD weight: `before` until `at_step`, then `after`.\n\n    Example: a concert ends at step 150 and suddenly everyone wants the platform:\n        od = {"0_5": switch(1, 5, 150), "0_6": 1, ...}\n    """\n    arr = np.full(steps + 1, float(before))\n    arr[at_step:] = float(after)\n    return arr\n\n\ndef ramp(start: float, end: float, steps: int = 300) -> np.ndarray:\n    """A time-varying OD weight that changes linearly from `start` to `end`."""\n    return np.linspace(float(start), float(end), steps + 1)\n\n\n# ----------------------------------------------------------------------------\n# Running a scenario\n# ----------------------------------------------------------------------------\ndef run(\n    od: Optional[Dict] = None,\n    demand: Optional[Dict[int, Dict]] = None,\n    route_choice: Optional[Dict] = None,\n    links: Optional[Dict[str, Dict]] = None,\n    network_dir: Path | str = DEFAULT_NETWORK,\n) -> "Result":\n    """Run one scenario and return a `Result`.\n\n    Args:\n        od: OD weights, e.g. {"0_5": 3, "0_6": 1, "1_5": 1, "1_6": 1}.\n            Values are *relative* per origin. A value can also be an array\n            (see `switch` / `ramp`) to make the weight change over time.\n            Missing pairs keep the value from sim_params.yaml.\n        demand: per-origin demand overrides, e.g. {0: {"peak_lambda": 60}}.\n        route_choice: path-finder overrides, e.g. {"beta": 0} (ignore congestion).\n        links: link overrides, e.g. {"2_3": {"width": 4}}.\n    """\n    gen = NetworkEnvGenerator()\n    gen.network_data = gen.load_network_data(str(network_dir))\n    params = gen.config["params"]\n    steps = params["simulation_steps"]\n\n    if route_choice:\n        params["path_finder"] = {**params.get("path_finder", {}), **route_choice}\n\n    od_flows = dict(gen.config.get("od_flows", {}))\n    for key, w in (od or {}).items():\n        o, d = _parse_key(key)\n        if np.ndim(w) and len(w) != steps + 1:\n            raise ValueError(f"OD {key}: time-varying weights need {steps + 1} values, got {len(w)}")\n        od_flows[(o, d)] = w\n\n    net = gen.create_network(\n        str(network_dir),\n        od_flows=od_flows,\n        demand_params_overrides={f"origin_{o}": p for o, p in (demand or {}).items()},\n        link_params_overrides=links or {},\n        verbose=False,\n    )\n    for t in range(1, steps):\n        net.network_loading(t)\n\n    return Result(network=net, steps=steps, unit_time=params["unit_time"], od=od_flows)\n\n\n# ----------------------------------------------------------------------------\n# Results\n# ----------------------------------------------------------------------------\n@dataclass\nclass Result:\n    network: object\n    steps: int\n    unit_time: float\n    od: dict\n    _pos: dict = field(default=None, repr=False)\n\n    # --- raw series ---------------------------------------------------------\n    @property\n    def t(self) -> np.ndarray:\n        return np.arange(self.steps)\n\n    def link(self, key: str):\n        return self.network.links[_parse_key(key)]\n\n    def density(self, key: str) -> np.ndarray:\n        """Density (ped/m^2) on a corridor, both directions added up."""\n        u, v = _parse_key(key)\n        d = self.network.links[(u, v)].density[: self.steps].astype(float)\n        if (v, u) in self.network.links:\n            d = d + self.network.links[(v, u)].density[: self.steps]\n        return d\n\n    def turned_away(self, origin: int) -> np.ndarray:\n        """Cumulative people who could NOT enter at a gate because the first\n        corridor was full (PedNStream drops this demand, it does not queue it)."""\n        node = self.network.nodes[origin]\n        out = next(l for l in node.outgoing_links if not str(l.link_id).startswith("virtual"))\n        wanted = np.concatenate([[0.0], np.cumsum(node.demand[: self.steps - 1])])\n        entered = out.cumulative_inflow[: self.steps]\n        return np.maximum(wanted - entered, 0)\n\n    def arrivals(self, dest: int) -> np.ndarray:\n        """Cumulative number of people who reached a destination."""\n        node = self.network.nodes[dest]\n        incoming = [l for l in node.incoming_links if not str(l.link_id).startswith("virtual")]\n        return sum(l.cumulative_outflow[: self.steps] for l in incoming)\n\n    # --- headline numbers ----------------------------------------------------\n    def kpis(self) -> pd.Series:\n        nodes = self.network.nodes\n        origins, dests = self.network.origin_nodes, self.network.destination_nodes\n        demand = sum(nodes[o].demand[: self.steps - 1].sum() for o in origins)\n        blocked = sum(self.turned_away(o)[-1] for o in origins)\n        arrived = {d: self.arrivals(d)[-1] for d in dests}\n        in_net = sum(l.num_pedestrians[: self.steps] for l in self.network.links.values())\n        dens = {c: self.density(c).max() for c in CORRIDORS}\n        worst = max(dens, key=dens.get)\n        k_crit = self.network.links[(2, 3)].k_critical\n        congested_minutes = sum((self.density(c) > k_crit).sum() for c in CORRIDORS) * self.unit_time / 60\n        return pd.Series(\n            {\n                "total demand (people)": round(demand),\n                "reached Platform X": round(arrived.get(5, 0)),\n                "reached City Exit Y": round(arrived.get(6, 0)),\n                "turned away at gates": round(blocked),\n                "still inside at end": round(demand - blocked - sum(arrived.values())),\n                "worst corridor": worst,\n                "peak density there (ped/m²)": round(dens[worst], 2),\n                "corridor-minutes congested": round(congested_minutes, 1),\n                "time spent inside (person-hours)": round(in_net.sum() * self.unit_time / 3600, 1),\n            }\n        )\n\n    def link_table(self) -> pd.DataFrame:\n        rows = []\n        for c in CORRIDORS:\n            u, v = _parse_key(c)\n            fwd = self.network.links[(u, v)]\n            bwd = self.network.links[(v, u)]\n            d = self.density(c)\n            rows.append(\n                {\n                    "corridor": c,\n                    "from → to": f"{NODE_NAMES[u]} → {NODE_NAMES[v]}",\n                    "width (m)": fwd.width,\n                    "length (m)": fwd.length,\n                    "people through": round(fwd.cumulative_inflow[self.steps - 1] + bwd.cumulative_inflow[self.steps - 1]),\n                    "peak density": round(d.max(), 2),\n                    "peak at minute": round(d.argmax() * self.unit_time / 60, 1),\n                }\n            )\n        return pd.DataFrame(rows).set_index("corridor")\n\n    def route_split(self) -> pd.DataFrame:\n        """How each OD pair splits over its routes (at the node where they diverge).\n\n        These are the route-choice probabilities PedNStream used in the last step.\n        """\n        rows = []\n        for od, paths in self.network.path_finder.od_paths.items():\n            if len(paths) == 1:\n                rows.append({"OD": f"{od[0]}_{od[1]}", "route": "-".join(map(str, paths[0])), "share": 1.0})\n                continue\n            k = next(i for i in range(1, min(map(len, paths))) if len({p[i] for p in paths}) > 1) - 1\n            node = self.network.nodes[paths[0][k]]\n            turn_p = getattr(node, "node_turn_probs", {}).get(od, {})\n            for p in paths:\n                up = p[k - 1] if k > 0 else -1\n                rows.append({\n                    "OD": f"{od[0]}_{od[1]}",\n                    "route": "-".join(map(str, p)),\n                    "share": round(turn_p.get((up, p[k + 1]), np.nan), 2),\n                })\n        return pd.DataFrame(rows)\n\n    # --- plots --------------------------------------------------------------\n    @property\n    def pos(self) -> dict:\n        if self._pos is None:\n            with open(DEFAULT_NETWORK / "node_positions.json") as f:\n                self._pos = {int(k): v for k, v in json.load(f).items()}\n        return self._pos\n\n    def plot_map(self, step: Optional[int] = None, ax=None, title: Optional[str] = None, vmax: float = 6):\n        """Network coloured by density. step=None -> the moment of highest density."""\n        if step is None:\n            total = sum(self.density(c) for c in CORRIDORS)\n            step = int(total.argmax())\n        if ax is None:\n            _, ax = plt.subplots(figsize=(6, 6))\n        segs, cols, widths = [], [], []\n        for c in CORRIDORS:\n            u, v = _parse_key(c)\n            segs.append([self.pos[u], self.pos[v]])\n            cols.append(self.density(c)[step])\n            widths.append(self.network.links[(u, v)].width * 3)\n        norm = Normalize(0, vmax)\n        lc = LineCollection(segs, cmap="RdYlGn_r", norm=norm, linewidths=widths, capstyle="round")\n        lc.set_array(np.array(cols))\n        ax.add_collection(lc)\n        for n, (x, y) in self.pos.items():\n            colour = "#2e7d32" if n in self.network.origin_nodes else "#c62828" if n in self.network.destination_nodes else "#424242"\n            marker = "s" if n in self.network.origin_nodes else "D" if n in self.network.destination_nodes else "o"\n            ax.scatter([x], [y], s=160, c=colour, marker=marker, zorder=3, edgecolor="white")\n            ax.annotate(NODE_NAMES[n], (x, y), xytext=(8, 8), textcoords="offset points", fontsize=9)\n        for c, (u, v) in zip(CORRIDORS, (_parse_key(c) for c in CORRIDORS)):\n            (x1, y1), (x2, y2) = self.pos[u], self.pos[v]\n            ax.text((x1 + x2) / 2, (y1 + y2) / 2, f"{self.density(c)[step]:.1f}", fontsize=8,\n                    ha="center", va="center", bbox=dict(boxstyle="round", fc="white", alpha=0.8, lw=0))\n        ax.set_aspect("equal")\n        ax.margins(0.25)\n        ax.axis("off")\n        plt.colorbar(lc, ax=ax, shrink=0.6, label="density (ped/m²)")\n        ax.set_title(title or f"Density at minute {step * self.unit_time / 60:.0f}")\n        return ax\n\n    def plot_links(self, corridors: Optional[Iterable[str]] = None, ax=None, title: Optional[str] = None):\n        """Density over time on each corridor. Dashed line = critical density."""\n        if ax is None:\n            _, ax = plt.subplots(figsize=(9, 4))\n        minutes = self.t * self.unit_time / 60\n        for c in corridors or CORRIDORS:\n            ax.plot(minutes, self.density(c), label=c)\n        ax.axhline(self.network.links[(2, 3)].k_critical, ls="--", c="grey", lw=1)\n        ax.set_xlabel("minute")\n        ax.set_ylabel("density (ped/m²)")\n        ax.legend(ncol=4, fontsize=8)\n        ax.set_title(title or "Density per corridor")\n        return ax\n\n    def plot_arrivals(self, ax=None, title: Optional[str] = None):\n        """Cumulative arrivals at each destination and people turned away at the gates."""\n        if ax is None:\n            _, ax = plt.subplots(figsize=(9, 4))\n        minutes = self.t * self.unit_time / 60\n        for d in self.network.destination_nodes:\n            ax.plot(minutes, self.arrivals(d), label=f"arrived at {NODE_NAMES[d]}")\n        for o in self.network.origin_nodes:\n            ax.plot(minutes, self.turned_away(o), ls=":", label=f"turned away at {NODE_NAMES[o]}")\n        ax.set_xlabel("minute")\n        ax.set_ylabel("people")\n        ax.legend(fontsize=8)\n        ax.set_title(title or "Arrivals and people turned away")\n        return ax\n\n    def dashboard(self, title: str = ""):\n        fig = plt.figure(figsize=(14, 6))\n        gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.4])\n        self.plot_map(ax=fig.add_subplot(gs[:, 0]))\n        self.plot_links(ax=fig.add_subplot(gs[0, 1]))\n        self.plot_arrivals(ax=fig.add_subplot(gs[1, 1]))\n        fig.suptitle(title)\n        fig.tight_layout()\n        return fig\n\n    def animate(self, every: int = 3):\n        """Animated map (returns HTML to display in a notebook)."""\n        from matplotlib.animation import FuncAnimation\n        from IPython.display import HTML\n\n        fig, ax = plt.subplots(figsize=(6, 6))\n        frames = range(0, self.steps, every)\n\n        def draw(step):\n            fig.clf()\n            self.plot_map(step=step, ax=fig.add_subplot(111))\n\n        anim = FuncAnimation(fig, draw, frames=frames, interval=120)\n        html = HTML(anim.to_jshtml())\n        plt.close(fig)\n        return html\n\n\n# ----------------------------------------------------------------------------\n# Comparing scenarios\n# ----------------------------------------------------------------------------\ndef compare(results: Dict[str, Result]) -> pd.DataFrame:\n    """Side-by-side KPI table for several scenarios."""\n    return pd.DataFrame({name: r.kpis() for name, r in results.items()})\n\n\ndef compare_maps(results: Dict[str, Result]):\n    """One density map per scenario, each at its own worst moment."""\n    n = len(results)\n    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 5.5))\n    for ax, (name, r) in zip(np.atleast_1d(axes), results.items()):\n        r.plot_map(ax=ax, title=name)\n    fig.tight_layout()\n    return fig\n\n\n# ----------------------------------------------------------------------------\n# Interactive playground (needs ipywidgets)\n# ----------------------------------------------------------------------------\ndef playground():\n    """Sliders for OD weights, demand and route choice. Re-runs on every change."""\n    import ipywidgets as w\n    from IPython.display import display, clear_output\n\n    def slider(desc, value, lo=0, hi=5, step=0.5):\n        return w.FloatSlider(value=value, min=lo, max=hi, step=step, description=desc,\n                             style={"description_width": "150px"}, layout=w.Layout(width="420px"))\n\n    s = {\n        "0_5": slider("A → Platform X", 1),\n        "0_6": slider("A → City Exit Y", 1),\n        "1_5": slider("B → Platform X", 1),\n        "1_6": slider("B → City Exit Y", 1),\n        "peakA": slider("Gate A peak (ped/step)", 40, 0, 80, 5),\n        "peakB": slider("Gate B peak (ped/step)", 40, 0, 80, 5),\n        "beta": slider("Congestion avoidance β", 2, 0, 10, 0.5),\n        "w23": slider("Corridor 2_3 width (m)", 2, 1, 6, 0.5),\n    }\n    out = w.Output()\n    status = w.HTML()\n\n    def share(a, b):\n        return f"{100 * a / (a + b):.0f}% / {100 * b / (a + b):.0f}%" if a + b else "no flow"\n\n    def update(*_):\n        v = {k: x.value for k, x in s.items()}\n        status.value = (\n            f"<b>Gate A</b> X/Y split: {share(v[\'0_5\'], v[\'0_6\'])} &nbsp;|&nbsp; "\n            f"<b>Gate B</b> X/Y split: {share(v[\'1_5\'], v[\'1_6\'])}"\n        )\n        res = run(\n            od={k: v[k] for k in ("0_5", "0_6", "1_5", "1_6")},\n            demand={0: {"peak_lambda": v["peakA"]}, 1: {"peak_lambda": v["peakB"]}},\n            route_choice={"beta": v["beta"]},\n            links={"2_3": {"width": v["w23"]}},\n        )\n        with out:\n            clear_output(wait=True)\n            res.dashboard()\n            plt.show()\n            display(res.kpis().to_frame("value"))\n\n    for x in s.values():\n        x.observe(update, names="value")\n    ui = w.VBox([\n        w.HTML("<h4>OD weights (relative, per gate)</h4>"),\n        w.HBox([w.VBox([s["0_5"], s["0_6"]]), w.VBox([s["1_5"], s["1_6"]])]),\n        status,\n        w.HTML("<h4>Demand, behaviour, infrastructure</h4>"),\n        w.HBox([w.VBox([s["peakA"], s["peakB"]]), w.VBox([s["beta"], s["w23"]])]),\n        out,\n    ])\n    display(ui)\n    update()\n', 'networks/station_square/sim_params.yaml': '# =====================================================================\n#  "Station Square" — workshop network for PedNStream\n# =====================================================================\n#\n#            6  City Exit (DEST Y)\n#            |\n#            |  4_6\n#            4  Mall Junction\n#            |  \\\n#   2_4      |    \\  3_4  (narrow diagonal)\n#  (wide)    |      \\\n#   0 ───────2 ───────3 ─────────── 5  Platform (DEST X)\n#  West     Plaza 2_3 Hall    3_5\n#  Gate        (NARROW)  |\n# (ORIGIN A)             |  1_3\n#                        1  South Gate (ORIGIN B)\n#\n#  Gate A enters at the Plaza (2), Gate B enters at the Hall (3).\n#  Routes (k_paths = 2):\n#    A -> X : 0-2-3-5   or 0-2-4-3-5\n#    A -> Y : 0-2-4-6   or 0-2-3-4-6\n#    B -> X : 1-3-5     (only one route!)\n#    B -> Y : 1-3-4-6   or 1-3-2-4-6\n#  A->X and B->Y can meet HEAD-ON in the narrow corridor 2_3.\n#\n#  Students normally only touch the `od_flows` block (via the notebooks),\n#  but every value here can be changed.\n# =====================================================================\n\nnetwork:\n  adjacency_matrix: [\n    [0, 0, 1, 0, 0, 0, 0],  # 0 West Gate     -> Plaza\n    [0, 0, 0, 1, 0, 0, 0],  # 1 South Gate    -> Hall\n    [1, 0, 0, 1, 1, 0, 0],  # 2 Plaza         -> West Gate, Hall, Mall\n    [0, 1, 1, 0, 1, 1, 0],  # 3 Hall          -> South Gate, Plaza, Mall, Platform\n    [0, 0, 1, 1, 0, 0, 1],  # 4 Mall Junction -> Plaza, Hall, City Exit\n    [0, 0, 0, 1, 0, 0, 0],  # 5 Platform\n    [0, 0, 0, 0, 1, 0, 0],  # 6 City Exit\n  ]\n  origin_nodes: [0, 1]\n  destination_nodes: [5, 6]\n\nsimulation:\n  seed: 42               # fixed seed -> every group sees the same demand\n  simulation_steps: 300  # 300 steps x 10 s = 50 minutes\n  unit_time: 10          # seconds per step\n  assign_flows_type: "classic"\n  path_finder:\n    k_paths: 2     # each OD pair gets its 2 shortest routes\n    temp: 5        # softmax temperature (lower = everyone picks the best route)\n    alpha: 1       # weight on route length\n    beta: 2        # weight on congestion (density)\n    omega: 1       # weight on capacity (width)\n    std_dev: 0     # no random noise in route choice\n\ndefault_link:\n  length: 50           # m\n  width: 3             # m\n  free_flow_speed: 1.1 # m/s\n  k_critical: 2        # ped/m^2  (above this the link is congested)\n  k_jam: 6             # ped/m^2  (jammed, nobody moves)\n  gamma: 0.01\n\nlinks:\n  "2_3": {width: 2}               # the narrow corridor = the bottleneck\n  "2_4": {width: 4}               # wide boulevard\n  "3_4": {width: 2, length: 70}   # narrow diagonal connector\n\n# Arrivals per time step at each gate (one peak at ~35% of the run).\n# NB: if the first corridor is full, PedNStream drops the excess (people are\n# \'turned away\'), it does not queue them outside the gate.\ndemand:\n  origin_0:\n    pattern: "single_peak"\n    base_lambda: 5\n    peak_lambda: 40\n    peak_position: 0.35\n  origin_1:\n    pattern: "single_peak"\n    base_lambda: 5\n    peak_lambda: 40\n    peak_position: 0.35\n\n# OD WEIGHTS — relative, not absolute! For each origin they are normalised\n# into "what share of the people entering here wants to go to each destination".\n#   0_5: 1, 0_6: 1  -> West Gate sends 50% to the Platform, 50% to City Exit\n#   0_5: 3, 0_6: 1  -> 75% / 25%\nod_flows:\n  "0_5": 1\n  "0_6": 1\n  "1_5": 1\n  "1_6": 1\n', 'networks/station_square/node_positions.json': '{"0": [0, 40], "1": [90, 0], "2": [40, 40], "3": [90, 40], "4": [40, 90], "5": [140, 40], "6": [40, 140]}\n'}

for _path, _text in _FILES.items():
    os.makedirs(os.path.dirname(_path) or ".", exist_ok=True)
    with open(_path, "w") as _f:
        _f.write(_text)

try:
    import pednstream  # noqa: F401
except ImportError:
    print("Installing PedNStream …")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/WaimenMak/PedNStream.git@d7ecbda107eca41feb3844985bb22e736b06dbe2"], check=True)
    importlib.invalidate_caches()

if "workshop_tools" in sys.modules:
    importlib.reload(sys.modules["workshop_tools"])

import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 90
from workshop_tools import run, compare, compare_maps, switch
_ok = run()
print("✅ Ready! Scroll down and continue.")

## 0 · Meet *Station Square*
```
            6  City Exit (Y)
            |
            4  Mall Junction
            |  \
   (wide)   |    \   (narrow diagonal)
            |      \
   0 ───────2 ───────3 ─────────── 5  Platform (X)
  West     Plaza  ↑  Hall
  Gate (A)    NARROW  |
                      1  South Gate (B)
```
* **Two entrances** (green squares): West Gate **A** enters at the Plaza, South Gate **B** enters at the Hall.
* **Two destinations** (red diamonds): the **Platform X** and the **City Exit Y**.
* Corridor **2_3** (Plaza–Hall) is only **2 m** wide.
* A rush of people arrives at both gates, peaking around minute 17. The rush is **the same in every run**.

In [ ]:
#@title What is an OD weight? Try it {display-mode: "form"}
#@markdown OD weights are **relative**. Set two weights for Gate A and see what share of Gate A's people goes where.
weight_A_to_Platform = 3  #@param {type:"slider", min:0, max:10, step:1}
weight_A_to_CityExit = 1  #@param {type:"slider", min:0, max:10, step:1}
total = weight_A_to_Platform + weight_A_to_CityExit
if total == 0:
    print("Both weights are 0. PedNStream then has no preference and falls back to 50% / 50%.")
else:
    pX = weight_A_to_Platform / total
    print(f"Of everyone entering at Gate A:  {pX:.0%} → Platform X,   {1 - pX:.0%} → City Exit Y")
    print("How MANY people enter is set by the gate's demand; the weights only split them.")
    plt.figure(figsize=(6, 1.2)); plt.barh([0], [pX], color="#1565c0"); plt.barh([0], [1 - pX], left=[pX], color="#ef6c00")
    plt.xlim(0, 1); plt.yticks([]); plt.title("Gate A: Platform X (blue) / City Exit Y (orange)"); plt.show()

> **Quick check:** weights 10 & 10 versus 1 & 1. Same result or different? Try it above.

In [ ]:
#@title ▶ Run the baseline (all weights equal) {display-mode: "form"}
base = run()
base.dashboard("Baseline: every gate sends 50% to X and 50% to Y"); plt.show()
display(base.kpis().to_frame("baseline"))

**How to read this**
* **Map**: corridor colour = density (people per m²) at the busiest moment. Above **2** (dashed line) a corridor is *congested* and people slow down sharply.
* **Density per corridor**: when and where crowds build up and clear.
* **Arrivals**: people reaching each destination over time. *Turned away* = people who couldn't even enter because the first corridor was full.
* **time spent inside (person-hours)**: our main "how bad was it" number. **Lower is better.**

---
## Exercise 1 · Everyone to one destination
A train is delayed, and **everyone** heads to the Platform X. Or a concert is on in town and **everyone** heads to the City Exit Y.

In [ ]:
#@title ✋ **Predict first**: Exercise 1 { display-mode: "form" }
#@markdown Which corridor will be worst in each case? Which case is worse overall?
my_prediction = ""  #@param {type:"string"}
print("Saved your prediction:", my_prediction or "(empty, please type one above and run again)")

In [ ]:
#@title ▶ Run Exercise 1 {display-mode: "form"}
all_x = run(od={"0_5": 1, "0_6": 0, "1_5": 1, "1_6": 0})
all_y = run(od={"0_5": 0, "0_6": 1, "1_5": 0, "1_6": 1})
display(compare({"baseline": base, "all to X": all_x, "all to Y": all_y}))
compare_maps({"baseline": base, "all to X": all_x, "all to Y": all_y}); plt.show()
print("How 'all to X' travellers split over their routes:")
display(all_x.route_split())

**Discuss**
1. In *all to X*, people are turned away at **Gate B**. Why B and not A? *(Hint: count B's routes to X in the table.)*
2. The narrow corridor 2_3 is *not* the worst. Where did the crowd go instead, and why?

---
## Exercise 2 · Same totals, different mix ⭐
Both scenarios send **exactly half** of all travellers to X and half to Y. Only *who* goes where differs:

| scenario | Gate A goes to | Gate B goes to |
|---|---|---|
| **crossing** | Platform X (east) | City Exit Y (north) |
| **parallel** | City Exit Y (north) | Platform X (east) |

In [ ]:
#@title ✋ **Predict first**: Exercise 2 { display-mode: "form" }
#@markdown Will the two scenarios give (almost) the same result? If not, which is worse, and where?
my_prediction = ""  #@param {type:"string"}
print("Saved your prediction:", my_prediction or "(empty, please type one above and run again)")

In [ ]:
#@title ▶ Run Exercise 2 {display-mode: "form"}
crossing = run(od={"0_5": 1, "0_6": 0, "1_5": 0, "1_6": 1})
parallel = run(od={"0_5": 0, "0_6": 1, "1_5": 1, "1_6": 0})
display(compare({"baseline": base, "crossing": crossing, "parallel": parallel}))
compare_maps({"crossing": crossing, "parallel": parallel}); plt.show()
print("Crossing scenario, corridor by corridor:")
display(crossing.link_table())

**Discuss**
1. Destination totals are identical, yet one scenario costs much more. Explain why in one sentence, using *route overlap* or *counter-flow*.
2. Knowing only "50% go to the Platform", could you have predicted the congestion?

---
## Exercise 3 · Destinations that change over time
A train is announced, and from that moment people become much more likely to head to the Platform.

In [ ]:
#@title ✋ **Predict first**: Exercise 3 { display-mode: "form" }
#@markdown People already walking to the City Exit when the announcement comes: do they keep going or turn around? Does the timing of the announcement matter?
my_prediction = ""  #@param {type:"string"}
print("Saved your prediction:", my_prediction or "(empty, please type one above and run again)")

In [ ]:
#@title ▶ Run Exercise 3 {display-mode: "form"}
announcement_minute = 25  #@param {type:"slider", min:0, max:49, step:1}
times_more_likely_to_go_to_Platform = 6  #@param {type:"slider", min:1, max:10, step:1}
step = int(announcement_minute * 6)       # 1 step = 10 s
k = times_more_likely_to_go_to_Platform
announce = run(od={"0_5": switch(1, k, at_step=step), "0_6": 1,
                   "1_5": switch(1, k, at_step=step), "1_6": 1})
ax = announce.plot_links(["0_2", "2_3", "3_5", "2_4", "4_6"], title=f"Announcement at minute {announcement_minute}")
ax.axvline(announcement_minute, color="k", lw=2, alpha=0.4); plt.show()
display(compare({"baseline": base, f"announced at min {announcement_minute}": announce}))

**Discuss**
1. Look at the vertical line. Do the corridors react with a delay, or all at once?
2. PedNStream applies the **current** OD weights at every junction, so people already inside "re-decide" at their next junction.
   Is that realistic for a loudspeaker announcement? For a departure board visible only at the entrance?
3. Try the announcement at minute **17** (the peak) and at minute **35**. Does *timing* matter as much as *size*?

---
## Exercise 4 · Do people dodge the jam?
People choose between two routes using **distance**, **crowding** and **width**.
**β (beta)** is how strongly people avoid crowds: 0 = they ignore crowding, 10 = they avoid it strongly.

In [ ]:
#@title ✋ **Predict first**: Exercise 4 { display-mode: "form" }
#@markdown For the bad crossing pattern, will people avoiding crowds (high β) fix the problem? Will a wider corridor?
my_prediction = ""  #@param {type:"string"}
print("Saved your prediction:", my_prediction or "(empty, please type one above and run again)")

In [ ]:
#@title ▶ Run Exercise 4 {display-mode: "form"}
scenario = "crossing (A to X, B to Y)"  #@param ["baseline (everyone 50/50)", "all to Platform X", "all to City Exit Y", "crossing (A to X, B to Y)", "parallel (A to Y, B to X)"]
beta_crowd_avoidance = 8  #@param {type:"slider", min:0, max:10, step:0.5}
width_of_corridor_2_3_m = 2  #@param {type:"slider", min:1, max:6, step:0.5}
od = {
    "baseline (everyone 50/50)":        {"0_5": 1, "0_6": 1, "1_5": 1, "1_6": 1},
    "all to Platform X":                {"0_5": 1, "0_6": 0, "1_5": 1, "1_6": 0},
    "all to City Exit Y":               {"0_5": 0, "0_6": 1, "1_5": 0, "1_6": 1},
    "crossing (A to X, B to Y)":        {"0_5": 1, "0_6": 0, "1_5": 0, "1_6": 1},
    "parallel (A to Y, B to X)":        {"0_5": 0, "0_6": 1, "1_5": 1, "1_6": 0},
}[scenario]
default = run(od=od)
changed = run(od=od, route_choice={"beta": beta_crowd_avoidance}, links={"2_3": {"width": width_of_corridor_2_3_m}})
label = f"β={beta_crowd_avoidance}, 2_3 = {width_of_corridor_2_3_m} m"
display(compare({"default (β=2, 2_3 = 2 m)": default, label: changed}))
compare_maps({"default": default, label: changed}); plt.show()

**Discuss**
1. Raising β changes the *peak density* more than the *total time*. Why can't smart walkers fix a bad OD pattern on their own?
2. Which helps more here: better behaviour (β), a wider corridor, or a different OD mix (Exercise 2)?

---
## Wrap-up: complete these three sentences with your partner
1. An OD weight controls …
2. Two OD patterns with the same destination totals can behave differently because …
3. When OD weights change over time, the network …

➡ Next: **Part 2 · Playground**, then **Part 3 · Challenge**.